In [54]:
# Spark Session
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = (
    SparkSession
    .builder
    .appName("Reading and Parsing JSON Files/Data")
    .master("local[*]")
    .getOrCreate()
)

spark

In [ ]:
SparkSession.stop()

In [55]:
# Read Single line JSON file

df_single = spark.read.format("json").option('multiline',True).load("datasets/order_multiline.json")

In [56]:
d=df_single.printSchema()
print(d)

root
 |-- customer: struct (nullable = true)
 |    |-- address: struct (nullable = true)
 |    |    |-- city: string (nullable = true)
 |    |    |-- country: string (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- name: string (nullable = true)
 |-- id: long (nullable = true)
 |-- membership: struct (nullable = true)
 |    |-- active: boolean (nullable = true)
 |    |-- level: string (nullable = true)
 |-- orders: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- date: string (nullable = true)
 |    |    |-- items: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- price: long (nullable = true)
 |    |    |    |    |-- product: string (nullable = true)
 |    |    |    |    |-- quantity: long (nullable = true)
 |    |    |-- orderId: string (nullable = true)

None


In [62]:
df_single.show()

+--------------------+----+------------+--------------------+
|            customer|  id|  membership|              orders|
+--------------------+----+------------+--------------------+
|{{London, UK}, al...|1001|{true, Gold}|[{2026-07-18, [{1...|
+--------------------+----+------------+--------------------+



In [100]:
df=(df_single.withColumn('level',df_single['membership'].level)
            .withColumn('membership_active',df_single['membership'].active)
).drop(df_single['membership'])
df=df.withColumn('order_date',explode(df['orders']))
df=df.withColumn('Orders_date',df['order_date'].date).drop(df['orders'])
df=df.withColumn('orders',df['order_date'].items).drop(df['order_date'])
df=df.withColumn('ordewr',explode(df['orders'])).drop(df['orders'])
df=df.withColumn('order_price',df['ordewr'].price)
df=df.withColumn('order_product',df['ordewr'].product)
df=df.withColumn('order_quantity',df['ordewr'].quantity).drop(df['ordewr'])
df=df.withColumn('cust_city',df['customer'].address.city)
df=df.withColumn('cust_country',df['customer'].address.country)
df=df.withColumn('cust_email',df['customer'].email)
df=df.withColumn('cust_name',df['customer.name']).drop(df['customer'])
df.show()
df.printSchema()

+----+-----+-----------------+-----------+-----------+-------------+--------------+---------+------------+-----------------+-------------+
|  id|level|membership_active|Orders_date|order_price|order_product|order_quantity|cust_city|cust_country|       cust_email|    cust_name|
+----+-----+-----------------+-----------+-----------+-------------+--------------+---------+------------+-----------------+-------------+
|1001| Gold|             true| 2026-07-18|       1200|       Laptop|             1|   London|          UK|alice@example.com|Alice Johnson|
|1001| Gold|             true| 2026-07-18|         25|        Mouse|             2|   London|          UK|alice@example.com|Alice Johnson|
|1001| Gold|             true| 2026-07-20|         80|     Keyboard|             1|   London|          UK|alice@example.com|Alice Johnson|
+----+-----+-----------------+-----------+-----------+-------------+--------------+---------+------------+-----------------+-------------+

root
 |-- id: long (nullab

In [58]:
df=(df_single.withColumn('skill1',df_single['skills'][0])
                .withColumn('skill2',df_single['skills'][1])
                .withColumn('skill3',df_single['skills'][2])
)
#df.withColumn('add', explode(df['address']))
df=df.withColumn('postcode',df['address'].postcode)

df.show()

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `skills` cannot be resolved. Did you mean one of the following? [`customer`, `id`, `membership`, `orders`]. SQLSTATE: 42703

In [ ]:
df=spark.createDataFrame(df_single['address'])

PySparkTypeError: [NOT_ITERABLE] Column is not iterable.

In [ ]:
df.show()

TypeError: 'Column' object is not callable

In [59]:
# Read Multiline JSON file

df_multi = spark.read.format("json").option("multiLine", True).load("datasets/order_multiline.json")

In [60]:
df_multi.printSchema()

root
 |-- customer: struct (nullable = true)
 |    |-- address: struct (nullable = true)
 |    |    |-- city: string (nullable = true)
 |    |    |-- country: string (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- name: string (nullable = true)
 |-- id: long (nullable = true)
 |-- membership: struct (nullable = true)
 |    |-- active: boolean (nullable = true)
 |    |-- level: string (nullable = true)
 |-- orders: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- date: string (nullable = true)
 |    |    |-- items: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- price: long (nullable = true)
 |    |    |    |    |-- product: string (nullable = true)
 |    |    |    |    |-- quantity: long (nullable = true)
 |    |    |-- orderId: string (nullable = true)



In [61]:
df_multi.show()

+--------------------+----+------------+--------------------+
|            customer|  id|  membership|              orders|
+--------------------+----+------------+--------------------+
|{{London, UK}, al...|1001|{true, Gold}|[{2026-07-18, [{1...|
+--------------------+----+------------+--------------------+



In [8]:
df = spark.read.format("text").load("data/input/order_singleline.json")

In [9]:
df.printSchema()

root
 |-- value: string (nullable = true)



In [11]:
df.show(truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                                              |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{"order_id":"O101","customer_id":"C001","order_line_items":[{"item_id":"I001","qty":6,"amount":102.45},{"item_id":"I003","qty":2,"amount":2.01}],"contact":[9000010000,9000010001]}|
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+



In [17]:
# With Schema

_schema = "customer_id string, order_id string, contact array<long>"

df_schema = spark.read.format("json").schema(_schema).load("data/input/order_singleline.json")

In [18]:
df_schema.show()

+-----------+--------+--------------------+
|customer_id|order_id|             contact|
+-----------+--------+--------------------+
|       C001|    O101|[9000010000, 9000...|
+-----------+--------+--------------------+



In [ ]:
root
 |-- contact: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_line_items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- amount: double (nullable = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- qty: long (nullable = true)

In [20]:
_schema = "contact array<string>, customer_id string, order_id string, order_line_items array<struct<amount double, item_id string, qty long>>"

In [21]:
df_schema_new = spark.read.format("json").schema(_schema).load("data/input/order_singleline.json")

In [22]:
df_schema_new.printSchema()

root
 |-- contact: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_line_items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- amount: double (nullable = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- qty: long (nullable = true)



In [23]:
df_schema_new.show()

+--------------------+-----------+--------+--------------------+
|             contact|customer_id|order_id|    order_line_items|
+--------------------+-----------+--------+--------------------+
|[9000010000, 9000...|       C001|    O101|[{102.45, I001, 6...|
+--------------------+-----------+--------+--------------------+



In [26]:
# Function from_json to read from a column

_schema = "contact array<string>, customer_id string, order_id string, order_line_items array<struct<amount double, item_id string, qty long>>"

from pyspark.sql.functions import from_json

df_expanded = df.withColumn("parsed", from_json(df.value, _schema))


In [27]:
df_expanded.printSchema()

root
 |-- value: string (nullable = true)
 |-- parsed: struct (nullable = true)
 |    |-- contact: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- customer_id: string (nullable = true)
 |    |-- order_id: string (nullable = true)
 |    |-- order_line_items: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- amount: double (nullable = true)
 |    |    |    |-- item_id: string (nullable = true)
 |    |    |    |-- qty: long (nullable = true)



In [28]:
df_expanded.show()

+--------------------+--------------------+
|               value|              parsed|
+--------------------+--------------------+
|{"order_id":"O101...|{[9000010000, 900...|
+--------------------+--------------------+



In [29]:
# Function to_json to parse a JSON string
from pyspark.sql.functions import to_json

df_unparsed = df_expanded.withColumn("unparsed", to_json(df_expanded.parsed))

In [30]:
df_unparsed.printSchema()

root
 |-- value: string (nullable = true)
 |-- parsed: struct (nullable = true)
 |    |-- contact: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- customer_id: string (nullable = true)
 |    |-- order_id: string (nullable = true)
 |    |-- order_line_items: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- amount: double (nullable = true)
 |    |    |    |-- item_id: string (nullable = true)
 |    |    |    |-- qty: long (nullable = true)
 |-- unparsed: string (nullable = true)



In [32]:
df_unparsed.select("unparsed").show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|unparsed                                                                                                                                                                               |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{"contact":["9000010000","9000010001"],"customer_id":"C001","order_id":"O101","order_line_items":[{"amount":102.45,"item_id":"I001","qty":6},{"amount":2.01,"item_id":"I003","qty":2}]}|
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+



In [36]:
# Get values from Parsed JSON

df_1 = df_expanded.select("parsed.*")

In [38]:
from pyspark.sql.functions import explode

df_2 = df_1.withColumn("expanded_line_items", explode("order_line_items"))

In [39]:
df_2.show()

+--------------------+-----------+--------+--------------------+-------------------+
|             contact|customer_id|order_id|    order_line_items|expanded_line_items|
+--------------------+-----------+--------+--------------------+-------------------+
|[9000010000, 9000...|       C001|    O101|[{102.45, I001, 6...|  {102.45, I001, 6}|
|[9000010000, 9000...|       C001|    O101|[{102.45, I001, 6...|    {2.01, I003, 2}|
+--------------------+-----------+--------+--------------------+-------------------+



In [48]:
df_3 = df_2.select("contact", "customer_id", "order_id", "expanded_line_items.*")

In [49]:
df_3.show()

+--------------------+-----------+--------+------+-------+---+
|             contact|customer_id|order_id|amount|item_id|qty|
+--------------------+-----------+--------+------+-------+---+
|[9000010000, 9000...|       C001|    O101|102.45|   I001|  6|
|[9000010000, 9000...|       C001|    O101|  2.01|   I003|  2|
+--------------------+-----------+--------+------+-------+---+



In [50]:
# Explode Array fields
df_final = df_3.withColumn("contact_expanded", explode("contact"))


In [51]:
df_final.printSchema()

root
 |-- contact: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- item_id: string (nullable = true)
 |-- qty: long (nullable = true)
 |-- contact_expanded: string (nullable = true)



In [53]:
df_final.drop("contact").show()

+-----------+--------+------+-------+---+----------------+
|customer_id|order_id|amount|item_id|qty|contact_expanded|
+-----------+--------+------+-------+---+----------------+
|       C001|    O101|102.45|   I001|  6|      9000010000|
|       C001|    O101|102.45|   I001|  6|      9000010001|
|       C001|    O101|  2.01|   I003|  2|      9000010000|
|       C001|    O101|  2.01|   I003|  2|      9000010001|
+-----------+--------+------+-------+---+----------------+



In [38]:
A=("""{
  "id": 1001,
  "customer": {
    "name": "Alice Johnson",
    "email": "alice@example.com",
    "address": {
      "city": "London",
      "country": "UK"
    }
  },
  "orders": [
    {
      "orderId": "ORD001",
      "date": "2026-07-18",
      "items": [
        {
          "product": "Laptop",
          "quantity": 1,
          "price": 1200
        },
        {
          "product": "Mouse",
          "quantity": 2,
          "price": 25
        }
      ]
    },
    {
      "orderId": "ORD002",
      "date": "2026-07-20",
      "items": [
        {
          "product": "Keyboard",
          "quantity": 1,
          "price": 80
        }
      ]
    }
  ],
  "membership": {
    "level": "Gold",
    "active": true
  }
}""")

spark.read.format('json').load(A)

26/07/18 14:22:25 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: {
  "id": 1001,
  "customer": {
    "name": "Alice Johnson",
    "email": "alice@example.com",
    "address": {
      "city": "London",
      "country": "UK"
    }
  },
  "orders": [
    {
      "orderId": "ORD001",
      "date": "2026-07-18",
      "items": [
        {
          "product": "Laptop",
          "quantity": 1,
          "price": 1200
        },
        {
          "product": "Mouse",
          "quantity": 2,
          "price": 25
        }
      ]
    },
    {
      "orderId": "ORD002",
      "date": "2026-07-20",
      "items": [
        {
          "product": "Keyboard",
          "quantity": 1,
          "price": 80
        }
      ]
    }
  ],
  "membership": {
    "level": "Gold",
    "active": true
  }
}.
java.lang.IllegalArgumentException: java.net.URISyntaxException: Relative path in absolute URI: {
  "id":%201001,%0A%20%20%22customer%22:%2

IllegalArgumentException: java.net.URISyntaxException: Relative path in absolute URI: {
  "id":%201001,%0A%20%20%22customer%22:%20%7B%0A%20%20%20%20%22name%22:%20%22Alice%20Johnson%22,%0A%20%20%20%20%22email%22:%20%22alice@example.com%22,%0A%20%20%20%20%22address%22:%20%7B%0A%20%20%20%20%20%20%22city%22:%20%22London%22,%0A%20%20%20%20%20%20%22country%22:%20%22UK%22%0A%20%20%20%20%7D%0A%20%20%7D,%0A%20%20%22orders%22:%20%5B%0A%20%20%20%20%7B%0A%20%20%20%20%20%20%22orderId%22:%20%22ORD001%22,%0A%20%20%20%20%20%20%22date%22:%20%222026-07-18%22,%0A%20%20%20%20%20%20%22items%22:%20%5B%0A%20%20%20%20%20%20%20%20%7B%0A%20%20%20%20%20%20%20%20%20%20%22product%22:%20%22Laptop%22,%0A%20%20%20%20%20%20%20%20%20%20%22quantity%22:%201,%0A%20%20%20%20%20%20%20%20%20%20%22price%22:%201200%0A%20%20%20%20%20%20%20%20%7D,%0A%20%20%20%20%20%20%20%20%7B%0A%20%20%20%20%20%20%20%20%20%20%22product%22:%20%22Mouse%22,%0A%20%20%20%20%20%20%20%20%20%20%22quantity%22:%202,%0A%20%20%20%20%20%20%20%20%20%20%22pric